# ComicBookGenerator ? Colab GPU

Run with Runtime ? Change runtime type ? GPU. This notebook clones the repository, preserves Colab's CUDA PyTorch, checks the accelerator, and runs the same API workflow as the local app.

In [ ]:
%cd /content
!rm -rf ComicBookGenerator
!git clone https://github.com/CallmeTruong/Comic_Studio.git ComicBookGenerator
%cd /content/ComicBookGenerator

In [ ]:
!nvidia-smi
import torch
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU runtime first.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Keep Colab's CUDA torch, but align Pillow with diffusers/transformers.
!pip install -q --upgrade --force-reinstall 'Pillow>=11.3.0'
!pip install -q fastapi uvicorn python-dotenv numpy diffusers transformers accelerate safetensors sentencepiece peft compel huggingface-hub langgraph langchain-core langchain-openai openai rembg fonttools onnxruntime-gpu
import PIL
print('Pillow:', PIL.__version__)
print('If Pillow was imported before this cell, restart the runtime and run all cells again.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
drive_models=Path('/content/drive/MyDrive/ComicBookGenerator/models')
print('Drive models:',drive_models if drive_models.exists() else 'not found; SDXL can download lazily')

In [ ]:
MODEL='sdxl_dreamshaper' # or 'sd15'
from core.model_registry import ensure_model
profile,model_path=ensure_model(MODEL)
print(profile['label'],model_path)

## Full workflow test

The next cells start FastAPI and call `/api/generate`. Output PNGs and job metadata are written to `/content/ComicBookGenerator/outputs/`.

In [ ]:
import os, subprocess, time, requests, json, sys
os.environ['PYTHONPATH']='/content/ComicBookGenerator'
log_path='/content/ComicBookGenerator/colab_backend.log'
log=open(log_path,'w',encoding='utf-8')
api_process=subprocess.Popen([sys.executable,'api.py'],cwd='/content/ComicBookGenerator',stdout=log,stderr=subprocess.STDOUT,text=True)
url='http://127.0.0.1:8000/api/capabilities'
deadline=time.time()+180
last_error=None
while time.time()<deadline:
    if api_process.poll() is not None:
        log.flush()
        print(open(log_path,encoding='utf-8').read())
        raise RuntimeError(f'Backend exited with code {api_process.returncode}; see {log_path}')
    try:
        response=requests.get(url,timeout=5)
        response.raise_for_status()
        print('Backend ready:',response.json())
        break
    except requests.RequestException as exc:
        last_error=exc
        time.sleep(3)
else:
    log.flush()
    print(open(log_path,encoding='utf-8').read())
    raise TimeoutError(f'Backend did not become ready: {last_error}')

In [ ]:
STEPS=20
payload={'prompt':'Write a short four-panel comic in natural English about one careful robot delivering one parcel. Use one setting, clear cause and effect, short dialogue, and a visual punchline.','model':MODEL,'steps':STEPS,'guidance':7.0,'lora':'','seed':'12345','pageCount':1}
events=[]
with requests.post('http://127.0.0.1:8000/api/generate',json=payload,stream=True,timeout=1800) as response:
 response.raise_for_status()
 for line in response.iter_lines(decode_unicode=True):
  if line and line.startswith('data: '):
   event=line[6:]; events.append(event); print(event)
Path('outputs/colab_events.json').write_text(json.dumps(events,ensure_ascii=False,indent=2),encoding='utf-8')

In [ ]:
from pathlib import Path
pages=sorted(Path('outputs').glob('comic_page_*.png'),key=lambda p:p.stat().st_mtime,reverse=True)
print(*[str(p.resolve()) for p in pages[:5]],sep='\n')